<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2025/2026  

<img src="../../img/portada.png" width="300"/>

---

#  Aprendizaje automático relacional  
### Primera Convocatoria – Junio 2026  

**Profesor:** Pedro Almagro Blanco

**Autores:**  
David Espina Apellaniz  
Marco Padilla Gómez  

**Fecha de entrega:** 8 de Junio de 2025  

</div>

# 04 - Random Forest: validación cruzada

En este notebook se evalúa el modelo **Random Forest** mediante validación cruzada.

El objetivo es obtener una medida más fiable del rendimiento medio del modelo y compararlo posteriormente con Decision Tree y kNN.

## Objetivo metodologico

La validacion cruzada de Random Forest comprueba si el modelo mantiene su rendimiento en distintas particiones del dataset. Esto es importante porque el modelo final debe generalizar, no solo funcionar bien en una division concreta.


## 1. Importación de librerías y configuración inicial

Se importan las funciones necesarias y se definen las rutas del proyecto.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import cross_val_score

sys.path.append("../../")

from src.data_loader import load_processed_dataset
from src.model_training import get_base_models

ROOT = Path("../../").resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Validación cruzada del modelo

Se aplica validación cruzada de 5 particiones usando `f1` como métrica principal.

### Lectura de resultados

Se usa `f1` como metrica principal por la naturaleza binaria de la tarea y el desequilibrio moderado de clases. La media y desviacion tipica se guardan para poder compararlas con los demas modelos.


In [2]:
df = load_processed_dataset(DATA_PROCESSED / "twitch_mature_features.csv")
X = df.drop(columns=["new_id", "mature"])
y = df["mature"]
model = get_base_models(random_state=42)["random_forest"]
scores = cross_val_score(model, X, y, scoring="f1", cv=5, n_jobs=-1)
cv_metrics = pd.DataFrame([{
    "model": "random_forest",
    "mean_f1": scores.mean(),
    "std_f1": scores.std(),
    "fold_scores": ",".join(f"{score:.4f}" for score in scores)
}])
cv_metrics

,model,mean_f1,std_f1,fold_scores
0,random_forest,0.282387,0.012123,"0.2941,0.2700,0.2889,0.2933,0.2656"


## 3. Guardado de métricas de validación cruzada

Se guardan las métricas para utilizarlas en la fase de comparación final.

In [3]:
cv_metrics.to_csv(RESULTS_DIR / "rf_cv_metrics.csv", index=False)

## 4. Conclusiones

La validación cruzada permite comprobar si Random Forest mantiene un rendimiento estable en distintos subconjuntos del dataset.